# 在新数据上验证调参结果

从 `tune_dtw.ipynb` 的结果里取出 baseline 上 ARI ≥ `MIN_ARI` 的 DTW 参数组，在一份没参与调参的新数据上重新打分，看它们是不是只碰巧适合 baseline。标准 DTW、欧氏、皮尔森一起跑作参照。

聚类和打分方式与调参时完全相同（徐秋凡的 k-medoids，`N_SEEDS` 个种子取平均）。在 `zrz/dtw` 文件夹里从上到下运行；只需要改第一格。

In [ ]:
# 设置：按需要修改
TUNING = 'outputs/tuning_baseline/results.csv'   # tune_dtw.ipynb 的结果
SELECT_SCHEME = 'true_cluster'                   # 按哪种标签挑参数组
MIN_ARI = 0.90                                   # baseline 上 ARI 达到多少的参数组拿来验证
DATA = '../simulation_new'                       # 新数据：含 processed/centered.csv 和 ground_truth.csv
OUTPUT = 'outputs/validation_medium'             # 结果保存位置
SCALE_FLOOR = 0.1                                # 与调参时相同
N_SEEDS = 20
SCHEMES = {'true_cluster': 6, 'shape_cluster': 5}
WORKERS = 4

In [ ]:
import csv, json
from multiprocessing import Pool
from pathlib import Path
from IPython.display import Markdown, display

from dtw import read_matrix, prepare_expression, preprocessing_config
from support.tuning import effective_key, evaluate, rank_key, command, load_results, append_results

tuned = [r for r in load_results(Path(TUNING)) if r['scheme'] == SELECT_SCHEME]
baseline = {r['key']: r for r in tuned}
chosen = [r for r in tuned if r['method'] == 'dtw' and r['ari_mean'] >= MIN_ARI]
references = [r for r in tuned if r['method'] != 'dtw' or r['name'] == 'standard DTW']
print(f'baseline 上 {SELECT_SCHEME} ARI ≥ {MIN_ARI} 的 DTW 参数组：{len(chosen)} 组；另加参照 {len(references)} 个。')

data, out = Path(DATA), Path(OUTPUT)
ids, times, raw = read_matrix(data/'processed'/'centered.csv')
values = prepare_expression(raw, preprocessing_config(SCALE_FLOOR))
with (data/'ground_truth.csv').open(encoding='utf-8-sig', newline='') as f:
    table = {r['gene_id']: r for r in csv.DictReader(f)}
truth = {s: [table[g][s] for g in ids] for s in SCHEMES}
seeds = tuple(range(1, N_SEEDS + 1))
print(f'新数据：{len(ids)} 个基因 × {len(times)} 个时间点')

## 运行（可以随时中断，重新运行会接着算）

In [ ]:
out.mkdir(parents=True, exist_ok=True)
results = load_results(out/'results.csv')
done = {r['key'] for r in results}
jobs = [(r['method'], r['params'], values, times, truth, SCHEMES, seeds)
        for r in chosen + references if r['key'] not in done]
with Pool(max(1, WORKERS)) as pool:
    for k, rows in enumerate(pool.imap_unordered(evaluate, jobs), 1):
        append_results(out/'results.csv', rows)
        results += rows
        if k % 25 == 0 or k == len(jobs):
            print(f'{k}/{len(jobs)}', flush=True)
print('完成')

## 对比：baseline 上的成绩 vs 新数据上的成绩

In [ ]:
def spearman(a, b):
    """Rank correlation (average ranks for ties): does the baseline order hold on the new data?"""
    def ranks(v):
        order = sorted(range(len(v)), key=lambda i: v[i]); r = [0.0]*len(v); i = 0
        while i < len(v):
            j = i
            while j+1 < len(v) and v[order[j+1]] == v[order[i]]:
                j += 1
            for k in range(i, j+1):
                r[order[k]] = (i+j)/2 + 1
            i = j+1
        return r
    ra, rb = ranks(a), ranks(b); n = len(a)
    ma, mb = sum(ra)/n, sum(rb)/n
    cov = sum((x-ma)*(y-mb) for x, y in zip(ra, rb))
    sa, sb = sum((x-ma)**2 for x in ra)**.5, sum((y-mb)**2 for y in rb)**.5
    return cov/(sa*sb) if sa and sb else float('nan')

new = sorted((r for r in results if r['scheme'] == SELECT_SCHEME), key=rank_key)
base_rank = {r['key']: i for i, r in enumerate(sorted(tuned, key=rank_key), 1)}
dtw_new = [r for r in new if r['method'] == 'dtw' and r['name'] != 'standard DTW']
rho = spearman([baseline[r['key']]['ari_mean'] for r in dtw_new], [r['ari_mean'] for r in dtw_new])
lines = [f'# 新数据验证（{SELECT_SCHEME}）', '',
         f'新数据：`{DATA}`；参数组：baseline 上 ARI ≥ {MIN_ARI} 的 {len(chosen)} 组 + 参照。', '',
         f'baseline 排名与新数据排名的相关（Spearman）：**{rho:.2f}**。'
         '接近 1 表示 baseline 上好的参数在新数据上也好；接近 0 表示 baseline 上的排名在新数据上不成立。', '',
         '| 新数据排名 | 配置 | 开启参数 | 新数据 ARI | 新数据 ARI 最差 | 新数据 NMI | baseline ARI | 变化 | baseline 排名 |',
         '| ---: | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |']
for i, r in enumerate(new, 1):
    b = baseline[r['key']]
    lines.append(f"| {i} | {r['name']} | {r['n_params']} | {r['ari_mean']:.4f} | {r['ari_min']:.4f} | {r['nmi_mean']:.4f} | "
                 f"{b['ari_mean']:.4f} | {r['ari_mean']-b['ari_mean']:+.4f} | {base_rank[r['key']]} |")
best = next(r for r in new if r['method'] == 'dtw')
lines += ['', f"新数据上最好的 DTW 参数组：**{best['name']}**  ", f"`{command(best['params'], SCALE_FLOOR)}`"]
summary = '\n'.join(lines) + '\n'
display(Markdown(summary))

## 保存

In [ ]:
(out/'summary.md').write_text(summary, encoding='utf-8')
print('已保存：', out/'results.csv', out/'summary.md')